<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

In [2]:
#SPACEX API BROKEN > FUNCTIONS REPLACED BY RETRO-ENGINEERING RESULT FUNCTIONS FROM FINAL CSV INFOS

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [3]:
# Dictionnaire statique de correspondance ID -> Nom de fusée
booster_lookup = {
    '5e9d0d95eda69973a809d1ec': 'Falcon 9',
    '5e9d0d95eda69955f709d1eb': 'Falcon 1',
    '5e9d0d95eda69974db09d1ed': 'Falcon Heavy'
}

def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            BoosterVersion.append(booster_lookup.get(x, 'Unknown'))


From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [4]:
# --- Dictionnaire 1 : launchpad -> infos site ---
launchpad_lookup = {
    '5e9e4501f509094ba4566f84': {'name': 'CCAFS SLC 40', 'longitude': -80.577366, 'latitude': 28.561857},
    '5e9e4502f509092b78566f87': {'name': 'VAFB SLC 4E', 'longitude': -120.610829, 'latitude': 34.632093},
    '5e9e4502f509094188566f88': {'name': 'KSC LC 39A', 'longitude': -80.603956, 'latitude': 28.608058},
}

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            info = launchpad_lookup.get(x, {'name': 'Unknown', 'longitude': None, 'latitude': None})
            Longitude.append(info['longitude'])
            Latitude.append(info['latitude'])
            LaunchSite.append(info['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [5]:

payload_lookup = {
    '5eb0e4b7b6c3bb0006eeb1e7': (6104.959412, 'LEO'),  # valeur moyenne (NaN d'origine)
    '5eb0e4bab6c3bb0006eeb1ea': (525.0, 'LEO'),
    '5eb0e4bbb6c3bb0006eeb1ed': (677.0, 'ISS'),
    '5eb0e4bbb6c3bb0006eeb1ee': (500.0, 'PO'),
    '5eb0e4bbb6c3bb0006eeb1ef': (3170.0, 'GTO'),
    '5eb0e4bbb6c3bb0006eeb1f0': (3325.0, 'GTO'),
    '5eb0e4bbb6c3bb0006eeb1f1': (2296.0, 'ISS'),
    '5eb0e4bcb6c3bb0006eeb1f2': (1316.0, 'LEO'),
    '5eb0e4bcb6c3bb0006eeb1f3': (4535.0, 'GTO'),
    '5eb0e4bcb6c3bb0006eeb1f4': (4428.0, 'GTO'),
    '5eb0e4bcb6c3bb0006eeb1f5': (2216.0, 'ISS'),
    '5eb0e4bdb6c3bb0006eeb1f6': (2395.0, 'ISS'),
    '5eb0e4bdb6c3bb0006eeb1f7': (570.0, 'ES-L1'),
    '5eb0e4bdb6c3bb0006eeb1fa': (1898.0, 'ISS'),
    '5eb0e4beb6c3bb0006eeb1fb': (4707.0, 'GTO'),
    '5eb0e4beb6c3bb0006eeb1fc': (2477.0, 'ISS'),
    '5eb0e4beb6c3bb0006eeb1fd': (2034.0, 'LEO'),
    '5eb0e4beb6c3bb0006eeb1fe': (553.0, 'PO'),
    '5eb0e4beb6c3bb0006eeb1ff': (5271.0, 'GTO'),
    '5eb0e4bfb6c3bb0006eeb200': (3136.0, 'ISS'),
    '5eb0e4bfb6c3bb0006eeb201': (4696.0, 'GTO'),
    '5eb0e4bfb6c3bb0006eeb202': (3100.0, 'GTO'),
    '5eb0e4c0b6c3bb0006eeb205': (2257.0, 'ISS'),
    '5eb0e4c1b6c3bb0006eeb206': (4600.0, 'GTO'),
    '5eb0e4c1b6c3bb0006eeb207': (5500.0, 'GTO'),
    '5eb0e4c2b6c3bb0006eeb208': (9600.0, 'PO'),
    '5eb0e4c3b6c3bb0006eeb209': (2490.0, 'ISS'),
    '5eb0e4c3b6c3bb0006eeb20a': (5600.0, 'GTO'),
    '5eb0e4c3b6c3bb0006eeb20b': (5300.0, 'GTO'),
    '5eb0e4c3b6c3bb0006eeb20c': (6104.959412, 'LEO'),
    '5eb0e4c3b6c3bb0006eeb20d': (6070.0, 'GTO'),
    '5eb0e4c4b6c3bb0006eeb20e': (2708.0, 'ISS'),
    '5eb0e4c4b6c3bb0006eeb20f': (3669.0, 'GTO'),
    '5eb0e4c4b6c3bb0006eeb210': (9600.0, 'PO'),
    '5eb0e4c4b6c3bb0006eeb211': (6761.0, 'GTO'),
    '5eb0e4c4b6c3bb0006eeb212': (2910.0, 'ISS'),
    '5eb0e4c4b6c3bb0006eeb213': (475.0, 'SSO'),
    '5eb0e4c5b6c3bb0006eeb214': (4990.0, 'LEO'),
    '5eb0e4c5b6c3bb0006eeb215': (9600.0, 'PO'),
    '5eb0e4c5b6c3bb0006eeb216': (5200.0, 'GTO'),
    '5eb0e4c5b6c3bb0006eeb217': (3700.0, 'GTO'),
    '5eb0e4c5b6c3bb0006eeb218': (2205.0, 'ISS'),
    '5eb0e4c6b6c3bb0006eeb219': (9600.0, 'PO'),
    '5eb0e4c6b6c3bb0006eeb21a': (6104.959412, 'LEO'),
    '5eb0e4c6b6c3bb0006eeb21b': (4230.0, 'GTO'),
    '5eb0e4c7b6c3bb0006eeb21f': (6092.0, 'GTO'),
    '5eb0e4c7b6c3bb0006eeb220': (9600.0, 'PO'),
    '5eb0e4c7b6c3bb0006eeb221': (2760.0, 'ISS'),
    '5eb0e4c7b6c3bb0006eeb222': (350.0, 'HEO'),
    '5eb0e4c7b6c3bb0006eeb223': (3750.0, 'GTO'),
    '5eb0e4c8b6c3bb0006eeb226': (5383.85, 'GTO'),
    '5eb0e4c8b6c3bb0006eeb227': (2410.0, 'ISS'),
    '5eb0e4c8b6c3bb0006eeb228': (7076.0, 'GTO'),
    '5eb0e4c9b6c3bb0006eeb229': (9600.0, 'PO'),
    '5eb0e4c9b6c3bb0006eeb22a': (5800.0, 'GTO'),
    '5eb0e4c9b6c3bb0006eeb22b': (7060.0, 'GTO'),
    '5eb0e4c9b6c3bb0006eeb22c': (2800.0, 'SSO'),
    '5eb0e4c9b6c3bb0006eeb22d': (3000.0, 'GTO'),
    '5eb0e4c9b6c3bb0006eeb22e': (4000.0, 'SSO'),
    '5eb0e4cab6c3bb0006eeb22f': (2573.0, 'ISS'),
    '5eb0e4cab6c3bb0006eeb230': (4400.0, 'MEO'),
    '5eb0e4cab6c3bb0006eeb231': (9600.0, 'PO'),
    '5eb0e4cbb6c3bb0006eeb235': (12259.0, 'ISS'),
    '5eb0e4cbb6c3bb0006eeb237': (2482.0, 'ISS'),
    '5eb0e4cbb6c3bb0006eeb238': (13620.0, 'VLEO'),
    '5eb0e4ccb6c3bb0006eeb239': (1425.0, 'SSO'),
    '5eb0e4ceb6c3bb0006eeb24a': (2227.7, 'ISS'),
    '5eb0e4cfb6c3bb0006eeb24b': (6500.0, 'GTO'),
    '5eb0e4cfb6c3bb0006eeb24c': (15600.0, 'VLEO'),
    '5eb0e4cfb6c3bb0006eeb24d': (5000.0, 'ISS'),
    '5eb0e4cfb6c3bb0006eeb24e': (6800.0, 'GTO'),
    '5eb0e4cfb6c3bb0006eeb24f': (15400.0, 'VLEO'),
    '5eb0e4d0b6c3bb0006eeb250': (6104.959412, 'SO'),  # valeur moyenne (NaN d'origine)
    '5eb0e4d0b6c3bb0006eeb251': (15600.0, 'VLEO'),
    '5eb0e4d0b6c3bb0006eeb252': (15400.0, 'VLEO'),
    '5eb0e4d0b6c3bb0006eeb253': (1977.0, 'ISS'),
    '5eb0e4d0b6c3bb0006eeb254': (15600.0, 'VLEO'),
    '5eb0e4d1b6c3bb0006eeb255': (15400.0, 'VLEO'),
    '5eb0e4d1b6c3bb0006eeb256': (15400.0, 'VLEO'),
    '5eb0e4d1b6c3bb0006eeb257': (9525.0, 'ISS'),
    '5eb0e4d1b6c3bb0006eeb258': (15400.0, 'VLEO'),
    '5eb0e4d1b6c3bb0006eeb259': (1600.0, 'SSO'),
    '5eb0e4d2b6c3bb0006eeb25b': (6104.959412, 'GEO'),  # valeur moyenne (NaN d'origine)
    '5eb0e4d2b6c3bb0006eeb25c': (3880.0, 'MEO'),
    '5eb0e4d2b6c3bb0006eeb25e': (3681.0, 'MEO'),
    '5ed9859f1f30554030d45c3f': (15400.0, 'VLEO'),
    '5ef6a4600059c33cee4a829e': (15400.0, 'VLEO'),
    '5ef6a48e0059c33cee4a829f': (15400.0, 'VLEO'),
    '5ef6a4d50059c33cee4a82a1': (15400.0, 'VLEO'),
    '5ef6a4ea0059c33cee4a82a2': (15400.0, 'VLEO'),
}

def getPayloadData(data):
    for load in data['payloads']:
        if load:
            info = payload_lookup.get(load, (None, None))
            PayloadMass.append(info[0])
            Orbit.append(info[1])


From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [6]:
# --- Dictionnaire 2 : core_id -> Serial/Block/ReusedCount ---
core_lookup = {
    '5e9e289ef359185f2b3b2628': ('B0003', 1.0, 0),
    '5e9e289ef35918f39c3b262a': ('B0005', 1.0, 0),
    '5e9e289ff359180ae23b262d': ('B1003', 1.0, 0),
    '5e9e289ff3591829343b2630': ('B1006', 1.0, 0),
    '5e9e289ff3591878603b262f': ('B1005', 1.0, 0),
    '5e9e289ff3591884e03b262c': ('B0007', 1.0, 0),
    '5e9e289ff35918862c3b262e': ('B1004', 1.0, 0),
    '5e9e28a0f359184a683b2634': ('B1010', 1.0, 0),
    '5e9e28a0f359186e2e3b2632': ('B1008', 1.0, 0),
    '5e9e28a0f3591870a63b2631': ('B1007', 1.0, 0),
    '5e9e28a0f359187a3c3b2635': ('B1012', 1.0, 0),
    '5e9e28a0f3591885be3b2636': ('B1013', 1.0, 0),
    '5e9e28a0f35918b1bc3b2633': ('B1011', 1.0, 0),
    '5e9e28a1f35918233f3b2639': ('B1016', 1.0, 0),
    '5e9e28a1f3591842fa3b263c': ('B1017', 1.0, 0),
    '5e9e28a1f3591867753b263b': ('B1019', 1.0, 0),
    '5e9e28a1f35918683c3b263a': ('B1018', 1.0, 0),
    '5e9e28a1f359186d533b2638': ('B1015', 1.0, 0),
    '5e9e28a1f359188def3b263d': ('B1020', 1.0, 0),
    '5e9e28a2f35918077b3b263f': ('B1022', 2.0, 0),
    '5e9e28a2f359182d0b3b263e': ('B1021', 2.0, 1),
    '5e9e28a2f3591845c73b2640': ('B1023', 2.0, 1),
    '5e9e28a2f359187ee83b2644': ('B1028', 3.0, 0),
    '5e9e28a2f359187f273b2642': ('B1025', 2.0, 1),
    '5e9e28a2f35918b8243b2643': ('B1026', 2.0, 0),
    '5e9e28a3f3591801cf3b264b': ('B1036', 3.0, 1),
    '5e9e28a3f3591811f83b2648': ('B1032', 3.0, 1),
    '5e9e28a3f3591829dc3b2646': ('B1031', 3.0, 1),
    '5e9e28a3f3591856803b264a': ('B1035', 3.0, 1),
    '5e9e28a3f359186f3f3b2649': ('B1034', 3.0, 0),
    '5e9e28a3f3591878473b2647': ('B1030', 3.0, 0),
    '5e9e28a3f359189e3a3b2645': ('B1029', 3.0, 1),
    '5e9e28a4f359182d843b264e': ('B1038', 3.0, 1),
    '5e9e28a4f35918345e3b2652': ('B1043', 4.0, 1),
    '5e9e28a4f3591843103b2650': ('B1041', 4.0, 1),
    '5e9e28a4f3591845123b264f': ('B1040', 4.0, 1),
    '5e9e28a4f3591850cc3b264c': ('B1037', 3.0, 0),
    '5e9e28a4f359185cc03b2651': ('B1042', 4.0, 0),
    '5e9e28a4f3591884ee3b264d': ('B1039', 4.0, 1),
    '5e9e28a5f3591809c03b2658': ('B1048', 5.0, 4),
    '5e9e28a5f359181eed3b2657': ('B1047', 5.0, 2),
    '5e9e28a5f359182b023b2656': ('B1046', 5.0, 3),
    '5e9e28a5f3591833b13b2659': ('B1049', 5.0, 5),
    '5e9e28a5f359186cb73b2654': ('B1044', 4.0, 0),
    '5e9e28a5f35918863d3b2655': ('B1045', 4.0, 1),
    '5e9e28a6f35918513b3b265b': ('B1054', 5.0, 0),
    '5e9e28a6f359185c603b265a': ('B1050', 5.0, 0),
    '5e9e28a6f35918c0803b265c': ('B1051', 5.0, 5),
    '5e9e28a7f3591809313b2660': ('B1056', 5.0, 3),
    '5e9e28a7f3591817f23b2663': ('B1058', 5.0, 2),
    '5e9e28a7f359187afd3b2662': ('B1059', 5.0, 3),
    '5ef670f10059c33cee4a826c': ('B1060', 5.0, 2),
    '5f57c5440622a633027900a0': ('B1062', 5.0, 0),
}

def getCoreData(data):
    for core in data['cores']:
        cid = core['core']
        if cid is not None and cid in core_lookup:
            serial, block, reused_count = core_lookup[cid]
            Block.append(block)
            ReusedCount.append(reused_count)
            Serial.append(serial)
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        # Ces champs sont déjà présents localement, pas besoin d'API :
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])  # déjà le bon ID, aucune résolution nécessaire

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [7]:
spacex_url="https://api.spacexdata.com/v4/launches/past"

In [8]:
response = requests.get(spacex_url)

Check the content of the response


In [9]:
print(response.content)

b'<!DOCTYPE html>\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\n<head>\n\n<title>spacexdata.com | 525: SSL handshake failed</title>\n<meta charset="UTF-8" />\n<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />\n<meta http-equiv="X-UA-Compatible" content="IE=Edge" />\n<meta name="robots" content="noindex, nofollow" />\n<meta name="viewport" content="width=device-width,initial-scale=1" />\n<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />\n</head>\n<body>\n<div id="cf-wrapper">\n    <div id="cf-error-details" class="p-0">\n        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">\n            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bla

You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [10]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [11]:
response=requests.get(static_json_url)

In [12]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [13]:
data=pd.json_normalize(response.json())



Using the dataframe <code>data</code> print the first 5 rows


In [14]:
data.head(5)


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [15]:
data_save=data
data_save.columns

Index(['static_fire_date_utc', 'static_fire_date_unix', 'tbd', 'net', 'window',
       'rocket', 'success', 'details', 'crew', 'ships', 'capsules', 'payloads',
       'launchpad', 'auto_update', 'failures', 'flight_number', 'name',
       'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming',
       'cores', 'id', 'fairings.reused', 'fairings.recovery_attempt',
       'fairings.recovered', 'fairings.ships', 'links.patch.small',
       'links.patch.large', 'links.reddit.campaign', 'links.reddit.launch',
       'links.reddit.media', 'links.reddit.recovery', 'links.flickr.small',
       'links.flickr.original', 'links.presskit', 'links.webcast',
       'links.youtube_id', 'links.article', 'links.wikipedia', 'fairings'],
      dtype='object')

In [16]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
print(len(data))
# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]
print(len(data))

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])
print(len(data))

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date
print(len(data))

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(len(data))

107
95
95
95
94


In [17]:
data.head(5)

,rocket,payloads,launchpad,cores,flight_number,date_utc,date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}",6,2010-06-04T18:45:00.000Z,2010-06-04


* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [18]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [19]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [20]:
# Call getBoosterVersion
getBoosterVersion(data)

the list has now been update 


In [21]:
BoosterVersion[0:5]

['Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 1', 'Falcon 9']

we can apply the rest of the  functions here:


In [22]:
# Call getLaunchSite
getLaunchSite(data)

In [23]:
# Call getPayloadData
getPayloadData(data)

In [24]:
# Call getCoreData
getCoreData(data)

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [25]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [30]:
# Create a data from launch_dict
launch_df = pd.DataFrame(launch_dict)

Show the summary of the dataframe


In [31]:
# Show the head of the dataframe
launch_df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 1,NaN,None,Unknown,None None,1,False,False,False,None,NaN,NaN,None,NaN,NaN
1,2,2007-03-21,Falcon 1,NaN,None,Unknown,None None,1,False,False,False,None,NaN,NaN,None,NaN,NaN
2,4,2008-09-28,Falcon 1,NaN,None,Unknown,None None,1,False,False,False,None,NaN,NaN,None,NaN,NaN
3,5,2009-07-13,Falcon 1,NaN,None,Unknown,None None,1,False,False,False,None,NaN,NaN,None,NaN,NaN
4,6,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0.0,B0003,-80.577366,28.561857


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [32]:
# Hint data['BoosterVersion']!='Falcon 1'
data_falcon9 = launch_df[launch_df['BoosterVersion'] != 'Falcon 1']

Now that we have removed some values we should reset the FlgihtNumber column


In [33]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/pandas/core/indexing.py:1773: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
4,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0.0,B0003,-80.577366,28.561857
5,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0.0,B0005,-80.577366,28.561857
6,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0.0,B0007,-80.577366,28.561857
7,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,None,1.0,0.0,B1003,-120.610829,34.632093
8,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,None,1.0,0.0,B1004,-80.577366,28.561857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,86,2020-09-03,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,2,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2.0,B1060,-80.603956,28.608058
90,87,2020-10-06,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,3,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,2.0,B1058,-80.603956,28.608058
91,88,2020-10-18,Falcon 9,15400.000000,VLEO,KSC LC 39A,True ASDS,6,True,True,True,5e9e3032383ecb6bb234e7ca,5.0,5.0,B1051,-80.603956,28.608058
92,89,2020-10-24,Falcon 9,15400.000000,VLEO,CCAFS SLC 40,True ASDS,3,True,True,True,5e9e3033383ecbb9e534e7cc,5.0,2.0,B1060,-80.577366,28.561857


## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [34]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [35]:
# Calculate the mean value of PayloadMass column
mean_payload_mass = data_falcon9['PayloadMass'].mean()

# Replace the np.nan values with its mean value
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_payload_mass)

/home/jupyterlab/conda/envs/python/lib/python3.7/site-packages/ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


In [36]:

data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
